# Подготовка

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

Mounted at /content/drive


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path
import numpy as np
import torch.nn.functional as F

# ASR

In [3]:
import shutil

drive_repo_path = "/content/drive/MyDrive/GigaAM_repo"
repo_path = "/content/GigaAM"
shutil.copytree(drive_repo_path, repo_path, dirs_exist_ok=True)
%cd {repo_path}
!pip install -e .

print("Библиотека восстановлена")

/content/GigaAM
Obtaining file:///content/GigaAM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.8 MB/s eta 0:00:00
  Building editable for gigaa

Библиотека восстановлена


In [1]:
#!pip uninstall torch torchvision torchaudio -y
#!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!pip install transformers==4.44.0 sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import gigaam
import time

print("Загружаем GigaAM-e2e_ctc...")
start_load = time.time()
model_asr = gigaam.load_model("e2e_ctc").float()
print(f"Загружена за {time.time() - start_load:.2f} сек")

Загружаем GigaAM-e2e_ctc...


100%|███████████████████████████████████████| 422M/422M [00:08<00:00, 54.4MiB/s]
100%|███████████████████████████████████████| 235k/235k [00:00<00:00, 8.60MiB/s]


Загружена за 12.39 сек


In [3]:
model_asr.load_state_dict(torch.load("/content/drive/MyDrive/asr_model_weights.pth"))
model_asr.eval()

GigaAMASR(
  (preprocessor): FeatureExtractor(
    (featurizer): Sequential(
      (0): MelSpectrogram(
        (spectrogram): Spectrogram()
        (mel_scale): MelScale()
      )
      (1): SpecScaler()
    )
  )
  (encoder): ConformerEncoder(
    (pre_encode): StridingSubsampling(
      (conv): Sequential(
        (0): Conv1d(64, 768, kernel_size=(5,), stride=(2,), padding=(2,))
        (1): ReLU()
        (2): Conv1d(768, 768, kernel_size=(5,), stride=(2,), padding=(2,))
        (3): ReLU()
      )
    )
    (pos_enc): RotaryPositionalEmbedding()
    (layers): ModuleList(
      (0-15): 16 x ConformerLayer(
        (norm_feed_forward1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (feed_forward1): ConformerFeedForward(
          (linear1): Linear(in_features=768, out_features=3072, bias=True)
          (activation): SiLU()
          (linear2): Linear(in_features=3072, out_features=768, bias=True)
        )
        (norm_conv): LayerNorm((768,), eps=1e-05, elementwis

In [8]:
class ASRDataset(Dataset):
    def __init__(self, folder_path=None, txt_path=None, paths=None, labels=None):
        self.files = []

        if folder_path is not None and txt_path is not None:
          with open(txt_path, 'r', encoding='utf-8') as f:
              for line in f:
                  parts = line.strip().split(maxsplit=1)
                  if len(parts) == 2:
                      name, text = parts
                      wav_path = Path(folder_path) / f"{name}.wav"
                      if wav_path.exists():
                          self.files.append((str(wav_path), text))
        elif paths is not None and labels is not None:
            self.files = list(zip(paths, labels))
        else:
          raise ValueError("Нужно указать либо folder_path и txt_path, либо paths и labels")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, text = self.files[idx]
        return path, text

    @classmethod
    def from_lists(cls, paths, labels):
        return cls(paths=paths, labels=labels)


def collate_fn(batch):
    paths, texts = zip(*batch)
    return list(paths), list(texts)

dataset = ASRDataset(
    folder_path = "/content/drive/MyDrive/876/",
    txt_path = "/content/drive/MyDrive/876/Текст.txt"
)
dataset_loader_asr = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

In [6]:
from sklearn.model_selection import train_test_split

all_paths = [item[0] for item in dataset.files]
all_texts = [item[1] for item in dataset.files]

train_paths, temp_paths, train_texts, temp_texts = train_test_split(
    all_paths, all_texts, test_size=0.4, random_state=42
)

val_paths, test_paths, val_texts, test_texts = train_test_split(
    temp_paths, temp_texts, test_size=0.5, random_state=42
)

test_dataset_asr = ASRDataset.from_lists(test_paths, test_texts)
test_loader_asr = DataLoader(test_dataset_asr, batch_size=8, shuffle=False, collate_fn=collate_fn)

In [9]:
all_pred_texts =  []
pred_labels = []

with torch.no_grad():
    for paths, texts in test_loader_asr:

        for path, text in zip(paths, texts):
            pred_text = model_asr.transcribe(path)
            all_pred_texts.append(pred_text)
            pred_labels.append(Path(path).stem.split('_')[0])

# Датасет от ASR

In [4]:
OUTPUT = "/content/drive/MyDrive/876_augmented/ASR.txt"
folder_path = "/content/drive/MyDrive/876_augmented/"

with open(OUTPUT, 'w', encoding='utf-8') as fl:
  with torch.no_grad():
    for f in Path(folder_path).iterdir():
      if f.suffix in ['.wav', '.mp3']:
        pred_text = model_asr.transcribe(f)
        name = f.stem
        fl.write(f"{name} {pred_text}\n")

In [5]:
!cat /content/drive/MyDrive/876_augmented/ASR.txt | head -5

insult-threat_anger_threat_080 вот и тупая.
insult-threat_anger_threat_080_aug0 вот и ттба.
insult-threat_anger_threat_080_aug1 мо вот окин дбнво.
insult-threat_anger_threat_080_aug2 вот и тупая.
insult-threat_anger_threat_078 иди нахуй. понятно?


In [6]:
!wc -l /content/drive/MyDrive/876_augmented/ASR.txt

3504 /content/drive/MyDrive/876_augmented/ASR.txt


# RuBert

In [8]:
model_path = "/content/drive/MyDrive/rubert_tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_path)
encoder = AutoModel.from_pretrained(model_path)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

In [9]:
class RuBERTClassifier(nn.Module):
    def __init__(self, encoder, num_classes=7, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(312, num_classes)  # 312 — размер эмбеддингов rubert-tiny2

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Берём CLS-токен (как пулинг в GigaAM-Emo)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_embedding)
        logits = self.classifier(x)
        return logits

In [10]:
class SemanticDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [11]:
semantic_list = ['absent', 'command', 'illegible', 'insult-threat', 'other', 'question', 'statement']
semantic2id = {sem: i for i, sem in enumerate(semantic_list)}
id2semantic = {i: sem for sem, i in semantic2id.items()}

pred_labels_ids = [semantic2id[label] for label in pred_labels]

In [12]:
test_dataset = SemanticDataset(all_pred_texts, pred_labels_ids, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [13]:
model = RuBERTClassifier(encoder, num_classes=7)
model.load_state_dict(torch.load("/content/drive/MyDrive/rubert_semantic_best.pth"))
model.eval()

test_preds, test_true = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['label']
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_true.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_true, test_preds)
test_f1 = f1_score(test_true, test_preds, average='weighted')
print(f"\n=== Тест ===")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1: {test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=semantic_list))


=== Тест ===
Test Accuracy: 0.7771
Test F1: 0.7641

Classification Report:
               precision    recall  f1-score   support

       absent       0.92      0.92      0.92        13
      command       0.79      0.85      0.81        26
    illegible       1.00      0.18      0.30        17
insult-threat       0.50      0.83      0.62        23
        other       0.95      0.80      0.87        25
     question       0.84      0.76      0.80        21
    statement       0.84      0.90      0.87        41

     accuracy                           0.78       166
    macro avg       0.83      0.75      0.74       166
 weighted avg       0.82      0.78      0.76       166

